# 03 - SQL Insights

Este notebook demonstra SQL na prática usando SQLite em memória. A ideia é carregar os CSVs gerados pelo pipeline e executar consultas de negócio semelhantes às de `sql/queries_analiticas.sql`.

SQL é importante porque aproxima a análise do ambiente real de dados: validação, agregação, segmentação e priorização costumam acontecer em bancos relacionais ou camadas semânticas de BI.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 120)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

DADOS_RAW = ROOT / "dados" / "raw"
DADOS_PROCESSED = ROOT / "dados" / "processed"
DADOS_OUTPUTS = ROOT / "dados" / "outputs"
print(f"Raiz do projeto: {ROOT}")

import sqlite3

In [ ]:
clientes = pd.read_csv(DADOS_PROCESSED / "clientes_limpo.csv")
transacoes = pd.read_csv(DADOS_PROCESSED / "transacoes_limpo.csv")
categorias = pd.read_csv(DADOS_PROCESSED / "categorias_limpo.csv")
predicoes = pd.read_csv(DADOS_OUTPUTS / "predicoes_churn.csv")

conn = sqlite3.connect(":memory:")
clientes.to_sql("clientes", conn, index=False, if_exists="replace")
transacoes.to_sql("transacoes", conn, index=False, if_exists="replace")
categorias.to_sql("categorias", conn, index=False, if_exists="replace")
predicoes.to_sql("predicoes", conn, index=False, if_exists="replace")
print("Tabelas carregadas no SQLite em memória.")

In [ ]:
def sql(query):
    return pd.read_sql_query(query, conn)

## 1. Validação de nulos

A primeira etapa em SQL é validar campos críticos para garantir confiabilidade das análises.

In [ ]:
sql("""
SELECT
    SUM(CASE WHEN cliente_id IS NULL THEN 1 ELSE 0 END) AS clientes_sem_id,
    SUM(CASE WHEN renda_mensal IS NULL THEN 1 ELSE 0 END) AS renda_nula,
    SUM(CASE WHEN saldo_atual IS NULL THEN 1 ELSE 0 END) AS saldo_nulo,
    SUM(CASE WHEN churn_flag IS NULL THEN 1 ELSE 0 END) AS churn_nulo
FROM clientes;
""")

## 2. Volume e ticket médio por estado

In [ ]:
sql("""
SELECT
    c.estado,
    COUNT(DISTINCT c.cliente_id) AS total_clientes,
    COUNT(t.transacao_id) AS total_transacoes,
    ROUND(SUM(t.valor), 2) AS volume_financeiro,
    ROUND(AVG(t.valor), 2) AS ticket_medio
FROM clientes c
LEFT JOIN transacoes t ON c.cliente_id = t.cliente_id
GROUP BY c.estado
ORDER BY volume_financeiro DESC;
""")

**Por que importa:** essa visão mostra onde esté o maior volume financeiro e ajuda a priorizar leitura regional no Power BI.

## 3. Top categorias por gasto

In [ ]:
sql("""
SELECT
    cat.nome_categoria,
    cat.tipo_macro,
    ROUND(SUM(t.valor), 2) AS volume_financeiro,
    COUNT(*) AS total_transacoes
FROM transacoes t
JOIN categorias cat ON t.categoria_id = cat.categoria_id
GROUP BY cat.nome_categoria, cat.tipo_macro
ORDER BY volume_financeiro DESC
LIMIT 10;
""")

## 4. Saldo líquido mensal por cliente

Como as transações são simuladas, usamos `tipo` para separar entradas e saídas. Essa consulta é útil para avaliar pressão financeira.

In [ ]:
sql("""
SELECT
    cliente_id,
    substr(data, 1, 7) AS ano_mes,
    ROUND(SUM(CASE WHEN tipo = 'Credito' THEN valor ELSE -valor END), 2) AS saldo_liquido_mensal
FROM transacoes
GROUP BY cliente_id, substr(data, 1, 7)
ORDER BY cliente_id, ano_mes
LIMIT 20;
""")

## 5. Score de risco por razão gasto/renda

In [ ]:
sql("""
WITH gasto_cliente AS (
    SELECT cliente_id, SUM(valor) AS gasto_total
    FROM transacoes
    WHERE tipo <> 'Credito'
    GROUP BY cliente_id
)
SELECT
    c.cliente_id,
    c.estado,
    c.perfil_risco,
    ROUND(c.renda_mensal, 2) AS renda_mensal,
    ROUND(g.gasto_total, 2) AS gasto_total,
    ROUND(g.gasto_total / NULLIF(c.renda_mensal, 0), 2) AS razao_gasto_renda
FROM clientes c
JOIN gasto_cliente g ON c.cliente_id = g.cliente_id
ORDER BY razao_gasto_renda DESC
LIMIT 15;
""")

## 6. Churn por perfil de risco

In [ ]:
sql("""
SELECT
    perfil_risco,
    COUNT(*) AS total_clientes,
    ROUND(AVG(churn_flag), 4) AS taxa_churn
FROM clientes
GROUP BY perfil_risco
ORDER BY taxa_churn DESC;
""")

## 7. Clientes prioritários para retenção

In [ ]:
sql("""
SELECT
    cliente_id,
    nome,
    estado,
    perfil_risco,
    renda_mensal,
    saldo_atual,
    prob_churn,
    risco,
    recomendacao
FROM predicoes
ORDER BY prob_churn DESC
LIMIT 20;
""")

## 8. Análise por canal de transação

In [ ]:
sql("""
SELECT
    canal,
    COUNT(*) AS total_transacoes,
    ROUND(SUM(valor), 2) AS volume_financeiro,
    ROUND(AVG(valor), 2) AS ticket_medio
FROM transacoes
GROUP BY canal
ORDER BY volume_financeiro DESC;
""")

## Conclusão

SQL ajuda a transformar arquivos em perguntas de negócio: onde esté o volume, quais clientes priorizar, quais perfis têm maior churn e quais canais concentram movimentação. Esse notebook mostra a camada analítica antes do Power BI.